In [35]:
from openai import OpenAI

openai_client = OpenAI()

Cargamos los documentos de github

In [36]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

Creamos el indice de busqueda

In [37]:
from minsearch import AppendableIndex
index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

E implementamos un RAG tradicional

In [38]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

Hacemos la plantilla del prompt del RAG

In [39]:
import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

Hacemos una query y construimos el prompt

In [40]:
question = "How do I create a dahsbord in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)


messages = [
    {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)
print(response.output_text)

The provided context does not contain information on how to create a dashboard in Evidently.


Ahora vamos a hacer este proceso agentico

Definimos el esquema de la herramienta de busqueda

In [41]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

Hacemos un primera prueba sin tools

In [42]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

response.usage.input_tokens

63

Ahora añadimos la tool, veremos que el numero de tokens es mayor porque la descripción de la tool se incluye en la entrada

In [43]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

110

Manejando la llamada a la tool

In [44]:
tool_call = response.output[0]
tool_call

ResponseFunctionToolCall(arguments='{"query":"create dashboard Evidently"}', call_id='call_gk2296m5OcBswHL8rqKsghLl', name='search', type='function_call', id='fc_0904c870567651da006ac8ddb48a3487d2a9944d7c4ec58614', async_=None, caller=None, namespace=None, status='completed')

Añadimos la llamada a los mensajes, parseamos los argumentos y buscamos con los argumentos

In [45]:
messages.append(tool_call)
tool_call.arguments

'{"query":"create dashboard Evidently"}'

In [46]:
arguments = json.loads(tool_call.arguments)
arguments

{'query': 'create dashboard Evidently'}

probamos con una llamada manual

In [47]:
search_results = search(query='create dashboard in Evidently')
search_results

[{'start': 0,
  'content': "<Check>\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n## What is a Dashboard?\n\nA Dashboard provides a clear view of your AI application performance. You can use it:\n\n- to track evaluation results across multiple experiments;\n- to track live production quality over time.\n\nEach Project has its own Dashboard. It's empty at first.\n\nTo populate it, you need to run an evaluation and **save at least one Report** to the Project. You can then choose values from Reports to plot.\n\n![](/images/dashboard_llm_light.png)\n\n## Dashboard Tabs\n\n<Check>\n  Multiple Tabs are available in **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\nYou can logically organize Panels within the same Dashboard into different Tabs.\n\n![](/images/dashboard_llm_tabs.gif)\n\n## **Dashboard Panels**\n\nA Panel is a visual element in the Dashboard that displays specified values in a single widget. Panels ca

Podemos hacer un unpack del diccionario de argumentos

In [48]:
search_results = search(**arguments)
search_results[:1]

[{'start': 0,
  'content': "<Check>\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n## What is a Dashboard?\n\nA Dashboard provides a clear view of your AI application performance. You can use it:\n\n- to track evaluation results across multiple experiments;\n- to track live production quality over time.\n\nEach Project has its own Dashboard. It's empty at first.\n\nTo populate it, you need to run an evaluation and **save at least one Report** to the Project. You can then choose values from Reports to plot.\n\n![](/images/dashboard_llm_light.png)\n\n## Dashboard Tabs\n\n<Check>\n  Multiple Tabs are available in **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\nYou can logically organize Panels within the same Dashboard into different Tabs.\n\n![](/images/dashboard_llm_tabs.gif)\n\n## **Dashboard Panels**\n\nA Panel is a visual element in the Dashboard that displays specified values in a single widget. Panels ca

Continuamos la conversacion

In [49]:
call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(search_results),
}

Añadimos la respuesta a la conversacion

In [50]:
messages.append(call_output)
messages

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nUse only facts from the knowledge base when answering.\nIMPORTANT: f you cannot find the answer, inform the user.\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard Evidently"}', call_id='call_gk2296m5OcBswHL8rqKsghLl', name='search', type='function_call', id='fc_0904c870567651da006ac8ddb48a3487d2a9944d7c4ec58614', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_gk2296m5OcBswHL8rqKsghLl',
  'output': '[{"start": 0, "content": "<Check>\\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\\n</Check>\\n\\n## What is a Dashboard?\\n\\nA Dashboard provides a clear view of your AI application performance. You can use it:\\n\\n- to track evaluation res

Obtenemos la respuesta final

In [51]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

3612

In [52]:
print(response.output_text)

To create a dashboard in Evidently, follow these steps:

1. **Create or Open a Project**: Ensure that you have a Project in Evidently. You need to run an evaluation and save at least one report to populate your dashboard.

2. **Access the Dashboard**: Each Project has its own dashboard, which starts empty.

3. **Add Tabs (Optional)**: If you want to organize your panels:
   - Enter "Edit" mode on the dashboard.
   - Click the plus sign to "add Tab."
   - You can select an empty tab or choose from pre-built templates.

4. **Add Panels**: You can add various types of panels (text, counters, line plots, etc.):
   - While in "Edit" mode, click on "Add Panel."
   - Configure the panel by selecting metrics from your reports.
   - You can also filter metrics using tags if you've added them to your reports.

5. **Save the Dashboard**: Once you’ve added your desired panels and tabs, save the dashboard to finish the setup.

For detailed guidance, you may refer to the UI or Python API documentati

Ahora vamos a verlo con salida estructurada

In [53]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [54]:
response = openai_client.responses.parse(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
    text_format=RAGResponse
)

response.usage.input_tokens

3836

In [55]:
rag_response = response.output_parsed
print(rag_response.answer)

## Creating a Dashboard in Evidently

To create a dashboard in Evidently, follow these steps:

### 1. **Setup Your Environment**
   - You need to sign up for [Evidently Cloud](https://app.evidently.cloud/signup) if you haven't already.
   - Create an **Organization** after logging in.
   - Make sure you have Reports saved to the Project, as Dashboards pull data from these Reports.

### 2. **Access the Dashboard**
   - Navigate to the Dashboard section within your project.
   - If this is your first time, your Dashboard will be empty.

### 3. **Add Tabs (optional)**
   - **Edit Mode**: Enter "Edit" mode on the Dashboard.
   - **Add a Tab**: Click the plus sign to add a new Tab.
     - You can create a new custom Tab by selecting "empty" and entering a name. 

### 4. **Adding Panels**
   - While in "Edit" mode, click on the "Add Panel" button to add new Panels to your Dashboard.
   - **Choose Panel Types**: You can add text panels, counters, line plots, pie charts, etc.
   - **Configure 

Funcion generica para el manejo de tools

In [56]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    else: 
        result = 'not found tool "{name}"'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

Instrucciones y pregunta

In [57]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

IMPORTANT: When you explore the knowledge base, make at least 3 different
searchers to make sure you explore the topic well.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""
question = "How do I create a dahsbord in Evidently?"

In [58]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

In [59]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)
message_history.extend(response.output)

In [60]:
for message in response.output:
    if message.type == 'function_call':
        print(f'executing {message.name}({message.arguments})...')
        tool_call_output = make_call(message)
        message_history.append(tool_call_output)

executing search({"query":"create dashboard"})...
executing search({"query":"dashboard features"})...
executing search({"query":"dashboard configuration"})...


In [61]:
message_history

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nIMPORTANT: When you explore the knowledge base, make at least 3 different\nsearchers to make sure you explore the topic well.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.\n\nOur knowledge base is entirely about Evidently, so you don't need to \ninclude the word 'evidently' in search results\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard"}', call_id='call_ZhaNjU6qSgywTIEShz9IXDIs', name='search', type='function_call', id='fc_0b78ea7005f360e8006ac8ddc13f7c87d29765f484c70009f2', async_=None, caller=None, namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"dashboard features"}', call_id='call_k3MpFIue4vuT941fxNfK1bEw', name='search', type='function_call', id='fc_0b

In [62]:
len(message_history)

8

In [ ]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

response.usage.input_tokens

Loop, cambiamos las instrucciones

In [63]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [64]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool],
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 1...
executing search({"query":"create dashboard"})...

iteraration number 2...
executing search({"query":"dashboard UI setup"})...
executing search({"query":"dashboard features"})...

iteraration number 3...
ASSISTANT: To create a dashboard, you can follow these general steps:

### 1. **Set Up Your Project**
   - Ensure you have connected to the Evidently Cloud and created a project. A dashboard is automatically empty upon creation and depends on reports for data.

### 2. **Add Tabs to Organize Panels**
   - You can have multiple tabs in your dashboard. To add a new tab:
     - Enter **Edit mode** on the dashboard (usually located at the top right).
     - Click the plus sign to add a tab, or select “empty” to create a custom tab.

### 3. **Adding Panels**
   - Panels are the visual elements within your dashboard. You have the option to add panels through the UI or using Python API.
   - To add a panel via the UI:
     - In **Edit mode**, click on the **"Add Panel"*

Añadimos mas tools

In [65]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}

In [66]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    elif name == 'add_entry':
        result = add_entry(**arguments)
    else: 
        result = 'not found tool "{name}"'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

Llamadas con tools multiples

In [67]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]
tools = [search_tool, add_entry_tool]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 1...
ASSISTANT: To assist you in understanding how to create a dashboard, I will first search for documentation specifically related to "creating a dashboard." This should provide a foundational overview of the process and any relevant instructions or guidelines.

I'll proceed with the search now.
executing search({"query":"creating a dashboard"})...

iteraration number 2...
ASSISTANT: From the initial search, I found comprehensive documentation on how to create a dashboard, including adding tabs and panels. To create a dashboard, you can use the user interface or the Python API, depending on your preference.

Next, I will analyze these results further to refine the information and search for additional queries that may provide more context or details, such as the specific steps for using the user interface to create a dashboard and best practices for panel configuration.
executing search({"query":"dashboard user interface steps"})...
executing search({"query":"dashb

Añadimos el contenido a la base de datos

In [69]:
message_history.append(
    {"role": "user", "content": "add this content to our database"}
)

In [70]:
while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 4...
executing add_entry({"filename":"docs/platform/dashboard_creation.mdx","title":"Creating a Dashboard","description":"How to create a dashboard using the user interface and Python API.","content":"### Steps to Create a Dashboard:\n\n1. **Connect to the Platform**: Ensure you have connected to Evidently Cloud and created a project.\n   \n2. **Adding Tabs**:\n   - Enter \"Edit\" mode on your dashboard.\n   - Click the \"+\" button to add a new tab. You can create an empty tab or select from pre-built templates.\n\n3. **Adding Panels**:\n   - In \"Edit\" mode, click the \"Add Panel\" button.\n   - Select the desired panel type (text, counter, pie chart, line plot, bar plot).\n   - Configure the panel by selecting metrics and setting visual parameters.\n\n4. **Preview and Save**: Use the preview function to review your panel and save it to the desired tab.\n\n### Useful Tips and Best Practices:\n- Ensure that your metrics are logged properly in the reports before add

In [71]:
index.docs[-1]

{'start': 0,
 'content': '### Steps to Create a Dashboard:\n\n1. **Connect to the Platform**: Ensure you have connected to Evidently Cloud and created a project.\n   \n2. **Adding Tabs**:\n   - Enter "Edit" mode on your dashboard.\n   - Click the "+" button to add a new tab. You can create an empty tab or select from pre-built templates.\n\n3. **Adding Panels**:\n   - In "Edit" mode, click the "Add Panel" button.\n   - Select the desired panel type (text, counter, pie chart, line plot, bar plot).\n   - Configure the panel by selecting metrics and setting visual parameters.\n\n4. **Preview and Save**: Use the preview function to review your panel and save it to the desired tab.\n\n### Useful Tips and Best Practices:\n- Ensure that your metrics are logged properly in the reports before adding them to the panels, as empty panels can occur if the required data is not available.\n- Familiarize yourself with the different panel types and their appropriate use cases to leverage the dashboard\

In [72]:
tools = [search_tool, add_entry_tool]

message_history = [
    {"role": "system", "content": instructions},
]

iteration_number = 1

# Q&A loop
while True:
    user_prompt = input('You:')
    if user_prompt.lower().strip() == 'stop':
        break

    message_history.append({"role": "user", "content": user_prompt})

    # tool-call loop
    while True:
        response = openai_client.responses.create(
            model='gpt-4o-mini',
            input=message_history,
            tools=tools,
        )

        print(f'iteraration number {iteration_number}...') 
        message_history.extend(response.output)

        has_function_calls = False

        for message in response.output:
            if message.type == 'function_call':
                print(f'executing {message.name}({message.arguments})...')
                tool_call_output = make_call(message)
                message_history.append(tool_call_output)
                has_function_calls = True

            if message.type == 'message':
                text = message.content[0].text
                print('ASSISTANT:', text)

        iteration_number = iteration_number + 1
        print()

        if not has_function_calls:
            break

iteraration number 1...
ASSISTANT: To provide you with relevant information, I will start by searching for general documentation on the features and functionality of the knowledge base. This will help me understand what specific topics are covered and allow me to give you a well-informed overview of the available content. 

I'll begin with a search for general information.
executing search({"query":"general features and functionality"})...

iteraration number 2...
ASSISTANT: The first search provided information about various metrics used for ranking and recommendation systems, highlighting key concepts such as precision, recall, F Beta scores, and Mean Average Precision (MAP). These metrics help evaluate the performance of models in retrieving and suggesting relevant items to users.

Next, I will analyze this information to identify areas that may require further exploration. I will perform two additional searches: one focused on specific methodologies of precision and recall, and ano

In [73]:
class Agent:

    def __init__(self, llm_client, model, instructions, tools):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name

        if name == 'search':
            result = search(**arguments)
        elif name == 'add_entry':
            result = add_entry(**arguments)
        else:
            result = 'not found tool "{name}"'

        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }   

    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions},
            ]

        message_history.append({"role": "user", "content": user_prompt})

        iteration_number = 0

        while True:
            response = self.llm_client.responses.create(
                model=self.model,
                input=message_history,
                tools=self.tools,
            )

            print(f'iteraration number {iteration_number}...') 
            message_history.extend(response.output)

            has_function_calls = False

            for message in response.output:
                if message.type == 'function_call':
                    print(f'executing {message.name}({message.arguments})...')
                    tool_call_output = self.make_call(message)
                    message_history.append(tool_call_output)
                    has_function_calls = True

                if message.type == 'message':
                    text = message.content[0].text
                    print('ASSISTANT:', text)

            iteration_number = iteration_number + 1
            print()

            if not has_function_calls:
                break

        return message_history        

    def qna(self):
        message_history = [
            {"role": "system", "content": instructions},
        ]

        iteration_number = 1

        # Q&A loop
        while True:
            user_prompt = input('You:')
            if user_prompt.lower().strip() == 'stop':
                break

            message_history = self.loop(user_prompt, message_history)

In [74]:
agent = Agent(
    llm_client=OpenAI(),
    model='gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

In [75]:
messages = agent.loop('evidently adshboards')

iteraration number 0...
ASSISTANT: To understand how to create or use dashboards, I will perform a search for "dashboards." This will help me find specific information related to the functionalities, features, or configurations of dashboards, allowing me to answer your query accurately.

Let's begin with the search.
executing search({"query":"dashboards"})...

iteraration number 1...
ASSISTANT: The search for "dashboards" yielded results that describe how to create and manage dashboards, add tabs, and integrate panels either through the user interface or via a Python API. These results specifically detail methods for adding various types of panels to the dashboard, including code examples for achieving this.

To further enhance the understanding of how these components fit into the overall dashboard functionality, I will now search for "adding dashboard panels" and "dashboard configuration." This will provide a more in-depth look at the process of configuring dashboards and the specifi

In [76]:
messages

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nMake 3 iterations:\n\n1) in the first iteration, perform one search\n2) in the second interation, analyze the results from the previous search\n   and perform 2 more searches\n3) synthesise the results into the output\n\nIMPORTANT: at each step, give an explanation of why you want to perform \nsearch for this particular search query. It should be 2-3 sentences explaining\nthe logic of your decision.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.\n\nOur knowledge base is entirely about Evidently, so you don't need to \ninclude the word 'evidently' in search results\n"},
 {'role': 'user', 'content': 'evidently adshboards'},
 ResponseOutputMessage(id='msg_0413671ea9639086006ac91bbec1cc87d29a5d22e0ee140689', content=[ResponseOutputText(annotations=[], text='To understand how to create or use 

Full qna

In [77]:
agent.qna()

Vamos con llamadas estructuradas, este ejemplo estructura todas las llamadas y no razona. Ademñas interfiere con la llamada a funciones

In [78]:
class SimpleStructuredAgent:

    def __init__(self, llm_client, model, instructions, tools, output_type=None):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools
        self.output_type = output_type

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name

        if name == 'search':
            result = search(**arguments)
        elif name == 'add_entry':
            result = add_entry(**arguments)
        else:
            result = 'not found tool "{name}"'

        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }   

    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions},
            ]

        message_history.append({"role": "user", "content": user_prompt})

        iteration_number = 0

        while True:
            if self.output_type:
                response = self.llm_client.responses.parse(
                    model=self.model,
                    input=message_history,
                    tools=self.tools,
                    text_format=self.output_type
                )
            else:
                response = self.llm_client.responses.create(
                    model=self.model,
                    input=message_history,
                    tools=self.tools,
                )

            print(f'iteraration number {iteration_number}...') 
            message_history.extend(response.output)

            has_function_calls = False

            for message in response.output:
                if message.type == 'function_call':
                    print(f'executing {message.name}({message.arguments})...')
                    tool_call_output = self.make_call(message)
                    message_history.append(tool_call_output)
                    has_function_calls = True

                if message.type == 'message':
                    text = message.content[0].text
                    print('ASSISTANT:', text)

            iteration_number = iteration_number + 1
            print()

            if not has_function_calls:
                break

        return message_history

In [79]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

Vamos con otra aproximacion, cambiamos las instrucciones

In [80]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

Heredamos del agente normal, hacemos primero todas las iteraciones pero la ultima pide la respuesta estructurada

In [81]:
class StructuredAgent1(Agent):

    def __init__(self, llm_client, model, instructions, tools, output_type):
        super().__init__(llm_client, model, instructions, tools)
        self.output_type = output_type

    def structured_loop(self, user_prompt, message_history=None):
        message_history = self.loop(user_prompt, message_history)
        response = self.llm_client.responses.parse(
            model=self.model,
            input=message_history,
            text_format=self.output_type
        )

        output = response.output_parsed
        return message_history, output

In [82]:
structure_agent_1 = StructuredAgent1(
    llm_client=OpenAI(),
    model='gpt-4o-mini',
    instructions=instructions,
    tools=[search_tool],
    output_type=RAGResponse
)

_, output = structure_agent_1.structured_loop('bashboards')

iteraration number 0...
ASSISTANT: It looks like there might be a typo in your query regarding "bashboards." I assume you meant "dashboards." To get relevant information, I'll perform a search for "dashboards" to find any documentation related to that topic. Dashboards are integral in data visualization and monitoring, so understanding their features and functionalities would be helpful. 

Let's start with the search.
executing search({"query":"dashboards"})...

iteraration number 1...
ASSISTANT: The search results returned relevant information regarding dashboards and their functionalities. Here's a summary of important aspects related to dashboards based on the documentation:

1. **Adding Panels**: You can add panels to your dashboards either through the user interface or via the Python API. This includes steps such as entering edit mode, clicking the "Add panel" button, and configuring the panel type and parameters.

2. **Panel Types**: There are multiple types of panels that you ca

In [83]:
print(output.answer)

### Dashboard Features and Customization

1. **Dashboard Creation & Configuration**:  
   - You can create dashboards via the user interface by entering an "Edit" mode and adding panels, or use the Python API.  

2. **Adding Tabs**:  
   - Organize the content by creating multiple tabs. New tabs are created on-the-fly when adding panels to them if they do not exist.  

3. **Panel Parameters**:  
   - Each panel can be customized with parameters such as title, size (full or half), values to display, and filters for specific report data.

### Types of Panels:

1. **Counter Panels**:  
   - Display a single value with optional text, such as counts or metrics.  
   - Examples include showing total counts or last values.  

2. **Plot Panels**:  
   - Visually represent data through line, bar, scatter, and histogram plots.  
   - Multiple series can be added to a single plot (e.g., multi-line charts).  

3. **Metric and Test Panels**:  
   - **Metric Panels** plot individual report values ov

Tercera aproximacion fake tool call para modelos que no soportan salida estructurada

In [84]:
def create_fake_tool(output_type, name="structure_result"):
    schema = output_type.model_json_schema()
    schema["type"] = "object"
    schema["additionalProperties"] = False

    structure_result_tool = {
        "type": "function",
        "name": name,
        "description": "Call this function when you're ready to show the final result as structured data.",
        "strict": True,
        "parameters": schema,
    }

    return structure_result_tool

In [86]:
class StructuredAgent2:

    FAKE_TOOL_NAME = 'structure_result'

    def __init__(self, llm_client, model, instructions, tools, output_type):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools + [create_fake_tool(output_type, self.FAKE_TOOL_NAME)]
        self.output_type = output_type

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name

        if name == 'search':
            result = search(**arguments)
        elif name == 'add_entry':
            result = add_entry(**arguments)
        else:
            result = 'not found tool "{name}"'

        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }   

    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions},
            ]

        message_history.append({"role": "user", "content": user_prompt})

        iteration_number = 0

        while True:
            response = self.llm_client.responses.create(
                model=self.model,
                input=message_history,
                tools=self.tools,
            )

            print('iteraration number {iteration_number}...') 
            message_history.extend(response.output)

            output = None

            for message in response.output:
                if message.type == 'function_call':
                    print('executing {message.name}({message.arguments})...')
                    if message.name == self.FAKE_TOOL_NAME:
                        output_ready = True
                        output_json = json.loads(message.arguments)
                        output = self.output_type.model_validate(output_json)
                        continue

                    tool_call_output = self.make_call(message)
                    message_history.append(tool_call_output)

                if message.type == 'message':
                    text = message.content[0].text
                    print("ASSISTANT:", text)

            iteration_number = iteration_number + 1
            print()

            if output:
                break

        return message_history, output

In [87]:
structure_agent_2 = StructuredAgent2(
    llm_client=OpenAI(),
    model='gpt-4o-mini',
    instructions=instructions,
    tools=[search_tool],
    output_type=RAGResponse
)

_, output = structure_agent_2.loop('bashboards')

iteraration number {iteration_number}...
ASSISTANT: It seems like you're looking for information related to "dashboards." This term can refer to various functionalities related to data visualization, reporting, or performance tracking. I'll start by searching for relevant documentation on dashboards to see what features and capabilities are covered.

Let's perform the first search now.
executing {message.name}({message.arguments})...

iteraration number {iteration_number}...
ASSISTANT: The first search has returned several relevant pieces of documentation focused on dashboards. Key points include:

1. **Adding Panels**: Instructions on how to add various types of panels (e.g., plots, distributions) to a dashboard, either through a user interface or using a Python API.
2. **Tabs**: Information about organizing panels into multiple tabs within a dashboard.
3. **Panel Types**: A detailed overview of different dashboard panel types, which allows users to visualize and monitor various metri

In [88]:
print(output.answer)

### Dashboard Overview

1. **What is a Dashboard?**
   - A dashboard provides a visual representation of evaluation results from experiments and aids in tracking application performance over time. It is initially empty and relies on saved Reports to populate it.

2. **Dashboard Configuration:**
   - Dashboards can include multiple **Tabs** to organize Panels. Users can add or delete Tabs as needed, and Panels can be added either through a Python API or a user interface.

3. **Panel Types:**
   - Panels can take various forms, including:
     - **Counters**: Display numeric values with optional text.
     - **Plots**: Include line plots, bar charts (grouped and stacked), and pie charts.
     - **Text Panels**: Useful for displaying titles or additional information.

4. **Adding Panels:**
   - The process to add Panels involves selecting a Metric from existing Reports, setting titles, choosing the size, and (optionally) tagging for filtering. Aggregations (e.g., sum, average, last) can b